"""
AIS Trajectory Prediction - Transformer Version (amélioré)

Conserve la même structure / fonctionnalités que le code d'origine :
 - Transformer encoder + autoregressive LSTM decoder
 - Early stopping + ReduceLROnPlateau scheduler
 - Visualisation des trajectoires
 - Clustering des routes (DBSCAN)
Améliorations :
 - typage, docstrings, robustesse device-aware
 - initialisation correcte des états du décodeur
 - sorties et conversions plus sûres (float32)
 - gestion des cas où aucune fenêtre n'est trouvée
 - logging simple + checkpoints
 - paramètres DBSCAN exposables
"""


In [9]:
%matplotlib inline
import os
import math
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm  # Version notebook de tqdm pour de jolies barres
from typing import Dict, Any, Tuple

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de base
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil utilisé : {device}")

# Reproductibilité
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Appareil utilisé : cuda


In [10]:
# --- HYPERPARAMÈTRES ---
SEQ_IN = 20
SEQ_OUT = 10
BATCH_SIZE = 64
LR = 1e-4          # Réduit pour éviter les NaNs
EPOCHS = 30
PATIENCE = 5
NUM_WORKERS = 0    # Toujours 0 sur Windows/Jupyter pour éviter les erreurs

# Architecture Modèle
HIDDEN = 128
N_HEADS = 8
N_LAYERS = 3
DROPOUT = 0.1
EMBED_VT = 16
INPUT_DIM_V2 = 8

In [11]:
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dl / 2.0) ** 2
    return 2.0 * R * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

def ade_fde(pred, true):
    N = pred.shape[0]
    ade_sum = 0.0
    fde_sum = 0.0
    for i in range(N):
        ade_sum += np.mean(haversine_np(pred[i,:,0], pred[i,:,1], true[i,:,0], true[i,:,1]))
        fde_sum += haversine_np(pred[i,-1,0], pred[i,-1,1], true[i,-1,0], true[i,-1,1])
    return ade_sum / N, fde_sum / N

class AISDataset(Dataset):
    def __init__(self, df: pd.DataFrame, seq_in=SEQ_IN, seq_out=SEQ_OUT):
        df = df.copy()
        
        # --- CORRECTIF NAN : Nettoyage drastique ---
        df = df.dropna(subset=["LAT", "LON", "SOG", "COG", "Heading", "BaseDateTime", "VesselType"])
        df = df[df["LAT"].between(-90, 90)]
        df = df[df["LON"].between(-180, 180)]
        df = df[(df["SOG"] >= 0) & (df["SOG"] < 80)] # Vitesse max réaliste
        # -------------------------------------------

        df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])
        df = df.sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)

        if df["VesselType"].dtype == object:
            df["VesselType"] = pd.to_numeric(df["VesselType"], errors="coerce").fillna(0).astype(int)
        
        self.vtype2idx = {v: i for i, v in enumerate(sorted(df["VesselType"].unique()))}
        self.idx2vtype = {i: v for v, i in self.vtype2idx.items()}
        df["VTIDX"] = df["VesselType"].map(self.vtype2idx).astype(int)

        self.seq_in = seq_in
        self.seq_out = seq_out
        windows = []

        for mmsi, g in df.groupby("MMSI"):
            g = g.reset_index(drop=True)
            if len(g) < (seq_in + seq_out): continue
            
            g['Delta_t'] = g['BaseDateTime'].diff().dt.total_seconds().fillna(0)
            arr = g[["LAT", "LON", "SOG", "COG", "Heading", "VTIDX", "Delta_t"]].to_numpy()
            
            max_start = len(g) - (seq_in + seq_out) + 1
            for i in range(max_start):
                windows.append((arr[i : i + seq_in], arr[i + seq_in : i + seq_in + seq_out]))

        if not windows: raise ValueError("Dataset vide après fenêtrage.")
        
        self.inp = np.array([w[0] for w in windows], dtype=np.float32)
        self.out = np.array([w[1] for w in windows], dtype=np.float32)

    def __len__(self): return len(self.inp)

    def __getitem__(self, idx):
        inp, out = self.inp[idx], self.out[idx]
        base_lat, base_lon = inp[-1, 0], inp[-1, 1]
        
        METER_PER_DEG_LAT = 111320.0
        meter_per_deg_lon = 40075000.0 * math.cos(math.radians(base_lat)) / 360.0

        x = np.stack([
            (inp[:, 0] - base_lat) * METER_PER_DEG_LAT,
            (inp[:, 1] - base_lon) * meter_per_deg_lon,
            inp[:, 2] / 10.0,
            np.sin(np.radians(inp[:, 3])), np.cos(np.radians(inp[:, 3])),
            np.sin(np.radians(inp[:, 4])), np.cos(np.radians(inp[:, 4])),
            inp[:, 6] / 300.0
        ], axis=1).astype(np.float32)

        y = np.stack([
            (out[:, 0] - base_lat) * METER_PER_DEG_LAT,
            (out[:, 1] - base_lon) * meter_per_deg_lon
        ], axis=1).astype(np.float32)

        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(y),
            "vtype": torch.tensor(int(inp[0, 5]), dtype=torch.long),
            "anchor": torch.tensor([base_lat, base_lon], dtype=torch.float32)
        }

In [12]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :].transpose(0, 1)
        return self.dropout(x)

class AIS_Transformer_V2(nn.Module):
    def __init__(self, input_dim=INPUT_DIM_V2, vtype_card=20, vtype_emb=EMBED_VT, hidden=HIDDEN, heads=N_HEADS, layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.vemb = nn.Embedding(vtype_card, vtype_emb)
        self.input_proj = nn.Linear(input_dim + vtype_emb, hidden)
        self.pos_encoder = PositionalEncoding(hidden, dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=heads, dim_feedforward=hidden*4, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.decoder_cell = nn.LSTMCell(2 + vtype_emb, hidden)
        self.out = nn.Linear(hidden, 2)

    def forward(self, x, vtype, out_len=SEQ_OUT, y_true=None, teacher_forcing_ratio=0.5):
        B, T, _ = x.shape
        v_emb = self.vemb(vtype)
        x_cat = torch.cat([x, v_emb.unsqueeze(1).expand(-1, T, -1)], dim=2)
        memory = self.encoder(self.pos_encoder(self.input_proj(x_cat)))
        
        h = memory[:, -1].contiguous()
        c = torch.zeros_like(h)
        prev = torch.zeros(B, 2, device=x.device)
        preds = []

        for t in range(out_len):
            if y_true is not None and random.random() < teacher_forcing_ratio:
                prev_input = y_true[:, t, :]
            else:
                prev_input = prev
            
            h, c = self.decoder_cell(torch.cat([prev_input, v_emb], dim=1), (h, c))
            delta = self.out(h)
            preds.append(delta.unsqueeze(1))
            prev = delta
        return torch.cat(preds, dim=1)

In [13]:
def evaluate(model, loader):
    model.eval()
    loss_fn = nn.MSELoss()
    total_loss = 0
    preds_all, trues_all, anchors_all = [], [], []

    with torch.no_grad():
        for batch in loader:
            x, y, v = batch["x"].to(device), batch["y"].to(device), batch["vtype"].to(device)
            pred = model(x, v, out_len=y.size(1), y_true=None)
            total_loss += loss_fn(pred, y).item() * x.size(0)
            
            preds_all.append(pred.cpu().numpy())
            trues_all.append(y.cpu().numpy())
            anchors_all.append(batch["anchor"].numpy())

    preds = np.concatenate(preds_all, axis=0)
    trues = np.concatenate(trues_all, axis=0)
    anchors = np.concatenate(anchors_all, axis=0)
    
    preds_ll = np.zeros_like(preds)
    trues_ll = np.zeros_like(trues)
    
    for i in range(len(preds)):
        lat, lon = anchors[i]
        mpd_lon = 40075000.0 * math.cos(math.radians(lat)) / 360.0
        preds_ll[i,:,0] = lat + preds[i,:,0] / 111320.0
        preds_ll[i,:,1] = lon + preds[i,:,1] / mpd_lon
        trues_ll[i,:,0] = lat + trues[i,:,0] / 111320.0
        trues_ll[i,:,1] = lon + trues[i,:,1] / mpd_lon

    ade, fde = ade_fde(preds_ll, trues_ll)
    return total_loss/len(loader.dataset), ade, fde

def train_notebook(df):
    full_ds = AISDataset(df)
    n = len(full_ds)
    train_size = int(0.8 * n)
    train_ds, test_ds = torch.utils.data.random_split(full_ds, [train_size, n - train_size])
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    vtype_card = len(full_ds.vtype2idx)
    model = AIS_Transformer_V2(vtype_card=vtype_card).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    
    # --- CORRECTION ICI : Suppression de verbose=True ---
    scheduler = ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
    # --------------------------------------------------
    
    best_fde = float('inf')
    patience_cnt = 0
    history = []
    
    print("--- Début de l'entraînement ---")
    for epoch in range(1, EPOCHS+1):
        model.train()
        total_train = 0
        tf = max(0.0, 1.0 - (epoch/10))

        # Barre de progression pour le Notebook
        pbar = tqdm(train_loader, desc=f"Ep {epoch}/{EPOCHS}", leave=False)
        for batch in pbar:
            x, y, v = batch["x"].to(device), batch["y"].to(device), batch["vtype"].to(device)
            opt.zero_grad()
            pred = model(x, v, out_len=y.size(1), y_true=y, teacher_forcing_ratio=tf)
            loss = nn.MSELoss()(pred, y)
            loss.backward()
            
            # --- CORRECTIF NAN : Gradient Clipping ---
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # -----------------------------------------
            
            opt.step()
            total_train += loss.item() * x.size(0)
            pbar.set_postfix({'loss': loss.item()})

        val_loss, ade, fde = evaluate(model, test_loader)
        scheduler.step(val_loss)
        
        history.append({'epoch': epoch, 'val_loss': val_loss, 'ade': ade, 'fde': fde})
        print(f"Ep {epoch} | Loss: {val_loss:.4f} | ADE: {ade:.1f}m | FDE: {fde:.1f}m")
        
        if fde < best_fde:
            best_fde = fde
            patience_cnt = 0
            torch.save(model.state_dict(), "best_model_notebook.pth")
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print("Early Stopping!")
                break
                
    model.load_state_dict(torch.load("best_model_notebook.pth"))
    return model, test_loader, full_ds.idx2vtype, history

In [14]:

# ==========================================
# 5. PONT : PRÉDICTIONS -> DATAFRAME POUR DBSCAN
# ==========================================

def extract_predictions_for_clustering(model, loader, idx2vtype):
    """Génère les prédictions et les convertit en DataFrame Lat/Lon global."""
    model.eval()
    results = []
    
    print("--- Extraction des prédictions pour Clustering ---")
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            x = batch["x"].to(device)
            v = batch["vtype"].to(device)
            anchors = batch["anchor"].numpy() # (B, 2)
            
            # Prédiction pure (sans teacher forcing)
            # On prédit la trajectoire future
            preds_offsets = model(x, v, out_len=SEQ_OUT, y_true=None).cpu().numpy() # (B, T, 2)
            v_types = v.cpu().numpy()
            
            B, T, _ = preds_offsets.shape
            
            for i in range(B):
                base_lat, base_lon = anchors[i]
                v_code_idx = v_types[i]
                real_v_code = idx2vtype.get(v_code_idx, v_code_idx) # Récupérer le vrai code AIS (ex: 70)
                
                mpd_lon = 40075000.0 * math.cos(math.radians(base_lat)) / 360.0
                
                # Convertir chaque point prédit en Lat/Lon global
                for t in range(T):
                    lat_pred = base_lat + preds_offsets[i, t, 0] / 111320.0
                    lon_pred = base_lon + preds_offsets[i, t, 1] / mpd_lon
                    
                    results.append({
                        "LAT": lat_pred,
                        "LON": lon_pred,
                        "VesselType": real_v_code,
                        "IsPrediction": True
                    })

    return pd.DataFrame(results)

In [15]:
def extract_predictions_for_clustering(model, loader, idx2vtype):
    model.eval()
    results = []
    
    print("--- Extraction des prédictions ---")
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            x = batch["x"].to(device)
            v = batch["vtype"].to(device)
            anchors = batch["anchor"].numpy()
            
            preds_offsets = model(x, v, out_len=SEQ_OUT, y_true=None).cpu().numpy()
            v_types = v.cpu().numpy()
            
            B, T, _ = preds_offsets.shape
            
            for i in range(B):
                base_lat, base_lon = anchors[i]
                v_code_idx = v_types[i]
                real_v_code = idx2vtype.get(v_code_idx, v_code_idx)
                mpd_lon = 40075000.0 * math.cos(math.radians(base_lat)) / 360.0
                
                for t in range(T):
                    lat_pred = base_lat + preds_offsets[i, t, 0] / 111320.0
                    lon_pred = base_lon + preds_offsets[i, t, 1] / mpd_lon
                    
                    results.append({
                        "LAT": lat_pred, "LON": lon_pred,
                        "VesselType": real_v_code, "IsPrediction": True
                    })
    return pd.DataFrame(results)

def generate_maritime_routes_dbscan(df, vessel_type_code, eps_meters=2000, min_samples=50):
    print(f"--- DBSCAN pour VesselType {vessel_type_code} ---")
    data = df[df["VesselType"] == vessel_type_code].copy()
    if len(data) == 0:
        print("Aucune donnée pour ce type.")
        return None

    coords = np.radians(data[["LAT", "LON"]].to_numpy())
    eps_rad = eps_meters / 6371000.0
    
    db = DBSCAN(eps=eps_rad, min_samples=min_samples, metric='haversine', algorithm='ball_tree')
    db.fit(coords)
    
    data["Cluster"] = db.labels_
    n_routes = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f"Routes trouvées : {n_routes} | Bruit : {list(db.labels_).count(-1)}")
    return data

def plot_maritime_routes(df_clustered, title="Routes"):
    if df_clustered is None: return
    plt.figure(figsize=(12, 8))
    noise = df_clustered[df_clustered["Cluster"] == -1]
    plt.scatter(noise["LON"], noise["LAT"], c='lightgray', s=1, alpha=0.3, label='Hors route')
    routes = df_clustered[df_clustered["Cluster"] != -1]
    if not routes.empty:
        sns.scatterplot(data=routes, x="LON", y="LAT", hue="Cluster", palette="tab10", s=5, linewidth=0, legend="full")
    plt.title(title)
    plt.xlabel("Longitude"); plt.ylabel("Latitude")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

In [ ]:
csv_file = "AIS_2024_01_04.csv"  # <--- METS TON CHEMIN ICI

if os.path.exists(csv_file):
    print("Lecture du CSV...")
    df_raw = pd.read_csv(csv_file)
    
    # Lancement de l'entraînement
    model, test_loader, idx2vtype, history = train_notebook(df_raw)
    
    # Afficher la courbe d'apprentissage
    plt.figure(figsize=(10,4))
    plt.plot([h['fde'] for h in history], label='FDE (mètres)')
    plt.title("Amélioration de la précision (FDE)")
    plt.xlabel("Epoch"); plt.ylabel("Erreur (m)")
    plt.legend(); plt.show()
else:
    print("Fichier CSV introuvable !")

Lecture du CSV...
--- Début de l'entraînement ---


Ep 1/30:   0%|          | 0/83350 [00:00<?, ?it/s]